In [1]:
import io
import zipfile
import requests
import frontmatter

In [2]:
url = "https://codeload.github.com/DataTalksClub/faq/zip/refs/heads/main"

resp = requests.get(url)

print(resp.status_code)

200


In [3]:
zf = zipfile.ZipFile(io.BytesIO(resp.content))

len(zf.infolist())

1490

In [4]:
repository_data = []

for file_info in zf.infolist():
    filename = file_info.filename.lower()

    if not filename.endswith(".md"):
        continue

    with zf.open(file_info) as f_in:
        content = f_in.read().decode("utf-8", errors="ignore")

        post = frontmatter.loads(content)

        data = post.to_dict()

        data["filename"] = filename

        repository_data.append(data)

In [5]:
len(repository_data)

1216

In [6]:
repository_data[0]

{'description': 'Review and process open FAQ PRs',
 'content': 'Go through all open pull requests one by one. For each PR:\n\n## 1. Show Details\n- PR number and title\n- Course and section (from PR body)\n- Related issue number\n- **ALWAYS check sort_order**: List files in target section (`ls _questions/<course>/<section>/`) to find highest number, verify PR uses next sequential\n- Full diff (use `gh pr diff <number>`)\n\n## 2. Check Against These Rules\n\n### Section Placement\n- **Kestra questions** → must be in `module-2` (workflow orchestration), NOT `general` or `module-1`\n- **Docker + Kestra** → still `module-2` (Kestra is primary topic)\n- **Docker-only** (pgAdmin, Postgres, etc.) → `module-1`\n\n### Relevance (Close If)\n- Basic Linux/SQL tutorials (vim installation, SQL JOIN concepts, etc.)\n- Generic programming not tied to course content\n- Already exists in DE zoomcamp when proposed for ML zoomcamp\n\n### Duplicates (Check Before Creating)\n- Search existing FAQs: `grep -

In [7]:
def read_repo_data(repo_owner, repo_name):

    prefix = "https://codeload.github.com"

    url = f"{prefix}/{repo_owner}/{repo_name}/zip/refs/heads/main"

    resp = requests.get(url)

    if resp.status_code != 200:
        raise Exception("Download failed")

    repository_data = []

    zf = zipfile.ZipFile(io.BytesIO(resp.content))

    for file_info in zf.infolist():

        filename = file_info.filename
        filename_lower = filename.lower()

        if not (
            filename_lower.endswith(".md")
            or filename_lower.endswith(".mdx")
        ):
            continue

        try:
            with zf.open(file_info) as f_in:

                content = f_in.read().decode(
                    "utf-8",
                    errors="ignore"
                )

                post = frontmatter.loads(content)

                data = post.to_dict()

                data["filename"] = filename

                repository_data.append(data)

        except Exception as e:
            print(e)

    zf.close()

    return repository_data

In [8]:
faq_data = read_repo_data(
    "DataTalksClub",
    "faq"
)

len(faq_data)

1216

In [9]:
my_repo = read_repo_data(
    "ezhilarasi0509",
    "AI-Agent-"
)

len(my_repo)

Exception: Download failed